리스트 내 관광지 별 시간대 json파일 저장

In [2]:
from pathlib import Path
import json
import time
import random
from datetime import datetime

import pandas as pd
import requests


# -----------------------------
# 경로 설정 (Excel 파일 및 컬럼명 변경)
# -----------------------------
INPUT_FILE = Path("카카오맵_기준_관광지_리스트_fin.xlsx")  # 엑셀 파일로 변경
RAW_DIR = Path("data/raw/visitor")
OUT_DIR = Path("data/processed")

DETAIL_CSV = OUT_DIR / "visitor_profiles.csv"
FAIL_CSV = OUT_DIR / "visitor_collection_failed.csv"

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://place.map.kakao.com/places/panel3/{}"

DAY_KEYS = {
    "monday": "monday_uv",
    "tuesday": "tuesday_uv",
    "wednesday": "wednesday_uv",
    "thursday": "thursday_uv",
    "friday": "friday_uv",
    "saturday": "saturday_uv",
    "sunday": "sunday_uv",
}

session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/130.0.0.0 Safari/537.36"
    ),
    "Referer": "https://place.map.kakao.com/",
})


# -----------------------------
# visitor 객체 찾기
# -----------------------------
def find_visitor(obj):
    if isinstance(obj, dict):
        v = obj.get("visitor")

        if isinstance(v, dict) and (
            "labels" in v
            or "weekly_uv_average" in v
            or any(k.endswith("_uv") for k in v)
        ):
            return v

        for value in obj.values():
            found = find_visitor(value)
            if found is not None:
                return found

    elif isinstance(obj, list):
        for value in obj:
            found = find_visitor(value)
            if found is not None:
                return found

    return None


# -----------------------------
# 관광지 목록 (엑셀 읽기 및 '관광지명' 사용)
# -----------------------------
# 엑셀 파일 읽기 (sheet_name 등 필요시 추가)
places = pd.read_excel(INPUT_FILE, dtype=str)
places.columns = places.columns.str.strip()

# '관광지명'과 '장소ID' 컬럼 사용 (엑셀 내 실제 장소 ID 컬럼명에 맞춰 수정 필요할 수 있음)
places = places[["관광지명", "장소ID"]].dropna(subset=["장소ID"])
places["관광지명"] = places["관광지명"].fillna("").str.strip()
places["장소ID"] = places["장소ID"].str.strip()
places = places[places["장소ID"] != ""].drop_duplicates("장소ID")

# 기존 CSV를 불러와 중단 후에도 결과 유지
if DETAIL_CSV.exists():
    old_df = pd.read_csv(DETAIL_CSV, encoding="utf-8-sig")
    records = old_df.to_dict("records")
else:
    records = []

if FAIL_CSV.exists():
    fail_df = pd.read_csv(FAIL_CSV, encoding="utf-8-sig")
    failures = fail_df.to_dict("records")
else:
    failures = []

# 성공한 장소라도 기존 CSV에 요일별 168개가 있는지 확인
existing = pd.DataFrame(records)

complete_ids = set()
if not existing.empty:
    for pid, group in existing.groupby("place_id"):
        daily = group[group["profile_day"].isin(DAY_KEYS.keys())]
        # 장소별 7개 요일 × 24시간이 존재할 때만 완료 처리
        if all(
            set(daily.loc[
                daily["profile_day"] == day, "hour"
            ].astype(int)) == set(range(24))
            for day in DAY_KEYS
        ):
            complete_ids.add(str(pid))


# -----------------------------
# 수집 및 변환
# -----------------------------
for i, row in places.reset_index(drop=True).iterrows():
    place_id = str(row["장소ID"])
    place_name = str(row["관광지명"])  # '검색어' 대신 '관광지명' 사용

    if place_id in complete_ids:
        print(f"[{i+1}/{len(places)}] 완료 데이터 존재: {place_name}")
        continue

    print(f"[{i+1}/{len(places)}] 수집: {place_name} ({place_id})")

    json_path = RAW_DIR / place_id / "visitor.json"
    json_path.parent.mkdir(parents=True, exist_ok=True)

    data = None
    error = None

    # 기존 JSON이 있으면 우선 재사용
    if json_path.exists():
        try:
            data = json.loads(json_path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            data = None

    # JSON이 없거나 visitor 데이터가 없으면 요청
    visitor = find_visitor(data) if data is not None else None

    if visitor is None:
        for attempt in range(3):
            try:
                response = session.get(
                    URL.format(place_id), timeout=20
                )

                if response.status_code != 200:
                    error = f"HTTP {response.status_code}"
                    if 400 <= response.status_code < 500:
                        break
                    time.sleep(2 * (attempt + 1))
                    continue

                data = response.json()
                visitor = find_visitor(data)

                if visitor is None:
                    error = "visitor 데이터 없음"
                    break

                # 원본 JSON 저장
                temp_path = json_path.with_suffix(".tmp")
                temp_path.write_text(
                    json.dumps(data, ensure_ascii=False, indent=2),
                    encoding="utf-8"
                )
                temp_path.replace(json_path)
                break

            except (requests.RequestException, ValueError) as e:
                error = str(e)
                time.sleep(2 * (attempt + 1))

    if visitor is None:
        failures = [
            f for f in failures
            if str(f.get("장소ID")) != place_id
        ]
        failures.append({
            "관광지명": place_name,  # '검색어' 대신 '관광지명' 사용
            "장소ID": place_id,
            "오류": error or "visitor 데이터 확인 실패",
            "시각": datetime.now().isoformat(timespec="seconds"),
        })
        pd.DataFrame(failures).to_csv(
            FAIL_CSV, index=False, encoding="utf-8-sig"
        )
        print(f"  실패: {error}")
        time.sleep(random.uniform(0.7, 1.4))
        continue

    labels = visitor.get("labels", list(range(24)))
    new_rows = []
    snapshot_time = visitor.get("updated_at")

    # 요일별 시간대 UV
    for day, key in DAY_KEYS.items():
        values = visitor.get(key)

        if not isinstance(values, list):
            continue

        for hour, uv in zip(labels, values):
            try:
                new_rows.append({
                    "place_id": place_id,
                    "place_name": place_name,
                    "profile_day": day,
                    "hour": int(hour),
                    "visitor_uv": float(uv),
                    "visitor_updated_at": snapshot_time,
                    "source_file": str(json_path),
                })
            except (TypeError, ValueError):
                continue

    # 주평균 시간대 UV도 별도 프로필로 저장
    values = visitor.get("weekly_uv_average")
    if isinstance(values, list):
        for hour, uv in zip(labels, values):
            try:
                new_rows.append({
                    "place_id": place_id,
                    "place_name": place_name,
                    "profile_day": "weekly_average",
                    "hour": int(hour),
                    "visitor_uv": float(uv),
                    "visitor_updated_at": snapshot_time,
                    "source_file": str(json_path),
                })
            except (TypeError, ValueError):
                continue

    # 동일 장소의 기존 결과를 새 응답으로 교체
    records = [
        r for r in records
        if str(r.get("place_id")) != place_id
    ]
    records.extend(new_rows)

    pd.DataFrame(records).to_csv(
        DETAIL_CSV, index=False, encoding="utf-8-sig"
    )

    # 실패 후 재수집 성공 시 실패 목록에서 제거
    failures = [
        f for f in failures
        if str(f.get("장소ID")) != place_id
    ]
    pd.DataFrame(
        failures,
        columns=["관광지명", "장소ID", "오류", "시각"]  # 컬럼명 수정 반영
    ).to_csv(FAIL_CSV, index=False, encoding="utf-8-sig")

    day_counts = (
        pd.DataFrame(new_rows)
        .query("profile_day != 'weekly_average'")
        ["profile_day"]
        .nunique()
    )

    print(
        f"  저장 완료: {len(new_rows)}개 행, "
        f"요일 데이터 {day_counts}/7"
    )

    time.sleep(random.uniform(0.7, 1.4))


# -----------------------------
# 최종 점검
# -----------------------------
final_df = pd.read_csv(DETAIL_CSV, encoding="utf-8-sig")

print("\n수집 종료")
print("장소 수:", final_df["place_id"].nunique())
print("총 행 수:", len(final_df))
print("저장 파일:", DETAIL_CSV.resolve())

daily = final_df[
    final_df["profile_day"].isin(DAY_KEYS.keys())
]

check = (
    daily.groupby(["place_id", "profile_day"])["hour"]
    .nunique()
)

print("\n요일별 시간대 개수 분포:")
print(check.value_counts().sort_index())

print("\n시간대가 24개 미만인 장소-요일 조합:")
print(check[check < 24].head(30))

print("\n실패 장소 수:", len(failures))
print("실패 로그:", FAIL_CSV.resolve())

[1/224] 수집: 2018 평창 동계올림픽대회 및 동계패럴림픽대회 기념관 (581024580)
  저장 완료: 192개 행, 요일 데이터 7/7
[2/224] 수집: 365세이프타운 (25045748)
  저장 완료: 192개 행, 요일 데이터 7/7
[3/224] 수집: BTS버스정류장 (1310990190)
  저장 완료: 192개 행, 요일 데이터 7/7
[4/224] 수집: DMZ박물관 (23794385)
  저장 완료: 192개 행, 요일 데이터 7/7
[5/224] 수집: DMZ생태평화공원 (1389012423)
  저장 완료: 192개 행, 요일 데이터 7/7
[6/224] 수집: 가령폭포 (8121115)
  저장 완료: 192개 행, 요일 데이터 7/7
[7/224] 수집: 가리산레포츠파크 (26916317)
  저장 완료: 192개 행, 요일 데이터 7/7
[8/224] 수집: 강릉선교장 (26979861)
  저장 완료: 192개 행, 요일 데이터 7/7
[9/224] 수집: 강릉솔향수목원 (25856200)
  저장 완료: 192개 행, 요일 데이터 7/7
[10/224] 수집: 강릉송정해변 (7960891)
  저장 완료: 192개 행, 요일 데이터 7/7
[11/224] 수집: 강릉안반데기관광농원 (937891746)
  저장 완료: 192개 행, 요일 데이터 7/7
[12/224] 수집: 강릉월화거리 (2058461010)
  저장 완료: 192개 행, 요일 데이터 7/7
[13/224] 수집: 강릉중앙시장 (18525909)
  저장 완료: 192개 행, 요일 데이터 7/7
[14/224] 수집: 강릉카페거리 (14862993)
  저장 완료: 192개 행, 요일 데이터 7/7
[15/224] 수집: 강릉해중공원 (714254383)
  저장 완료: 192개 행, 요일 데이터 7/7
[16/224] 수집: 강원감영 (8171435)
  저장 완료: 192개 행, 요일 데이터 7/7
[17/224] 수집: 강원도립화목원 (1104